# Drift Prediction - Burn-in Data (v4)

Predicts 96h/168h leakage/resistance/Vth from 0h/24h readings.

New in this version:
- tried weighting Fail/Borderline samples higher during training - didn't help, keeping the test
  in here anyway since it's a real finding (see that section for why)
- stacked ensemble (Ridge meta-model blending RF+GB) instead of just averaging them - small but
  real improvement across every target
- quantile regression for prediction intervals, not just point estimates - and the intervals
  actually widen on the step-change/unpredictable cases, which is exactly what you want
- bonus: tried a plain neural net and an LSTM too - couldn't test these myself (no GPU libs
  in the environment I built this in), should run fine on colab but numbers are unverified on my end

Upload the dataset csv before running. Check the CONFIG cell if pointing at a different/real dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold, cross_val_predict
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

## Config

In [ ]:
CSV_PATH = "burn_in_raw_dataset_10000.csv"

CHECKPOINTS = ['0h', '24h', '96h', '168h']
FEATURE_CHECKPOINTS = ['0h', '24h']
TARGET_CHECKPOINTS = ['96h', '168h']
METRICS = ['Leakage', 'Resistance', 'Vth']

STRESS_COL = 'Proxy_Stress'
STRESS_LEVEL_COL = 'Stress_Level'
PART_TYPE_COL = 'Part_Type'
LABEL_COL = 'Label'
TIMESTAMP_PREFIX = 'Timestamp_'

In [ ]:
# from google.colab import files
# uploaded = files.upload()

In [ ]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

In [ ]:
df[LABEL_COL].value_counts()

## Cleaning

Same as before - negative readings are impossible so treat as sensor faults, drop rows with
missing/bad targets, impute missing feature readings.

In [ ]:
reading_cols = [f'{m}_{cp}' for m in METRICS for cp in CHECKPOINTS]
for col in reading_cols:
    bad_count = (df[col] < 0).sum()
    if bad_count:
        print(f'{col}: {bad_count} bad sensor readings')
    df.loc[df[col] < 0, col] = np.nan

target_cols_raw = [f'{m}_{cp}' for m in METRICS for cp in TARGET_CHECKPOINTS]
feature_cols_raw = [f'{m}_{cp}' for m in METRICS for cp in FEATURE_CHECKPOINTS]

before = len(df)
df = df.dropna(subset=target_cols_raw).reset_index(drop=True)
print(f'dropped {before - len(df)} rows with missing/bad targets, {len(df)} remain')

## Timestamps -> elapsed hours

using the real gap between readings instead of assuming exactly 24/96/168

In [ ]:
for cp in CHECKPOINTS:
    df[f'{TIMESTAMP_PREFIX}{cp}'] = pd.to_datetime(df[f'{TIMESTAMP_PREFIX}{cp}'], format='ISO8601')
for cp in FEATURE_CHECKPOINTS[1:] + TARGET_CHECKPOINTS:
    df[f'elapsed_{cp}'] = (df[f'{TIMESTAMP_PREFIX}{cp}'] - df[f'{TIMESTAMP_PREFIX}0h']).dt.total_seconds() / 3600

df[['elapsed_24h','elapsed_96h','elapsed_168h']].describe()

## Train/test split

In [ ]:
idx_train, idx_test = train_test_split(df.index, test_size=0.2, random_state=42, stratify=df[LABEL_COL])
label_train = df.loc[idx_train, LABEL_COL]
label_test = df.loc[idx_test, LABEL_COL]
print(len(idx_train), 'train /', len(idx_test), 'test')

In [ ]:
imputer = SimpleImputer(strategy='median')
imputer.fit(df.loc[idx_train, feature_cols_raw])
df.loc[:, feature_cols_raw] = imputer.transform(df[feature_cols_raw])

## Features + targets

same setup as before - real elapsed hours for the slope, log leakage + resistance/vth deltas as targets

In [ ]:
df['Leakage_slope'] = (df['Leakage_24h'] - df['Leakage_0h']) / df['elapsed_24h']
df['Resistance_slope'] = (df['Resistance_24h'] - df['Resistance_0h']) / df['elapsed_24h']
df['Vth_slope'] = (df['Vth_24h'] - df['Vth_0h']) / df['elapsed_24h']

feature_cols_num = feature_cols_raw + [STRESS_LEVEL_COL, 'elapsed_24h', 'Leakage_slope', 'Resistance_slope', 'Vth_slope']
cat_cols = [STRESS_COL, PART_TYPE_COL]
X = pd.get_dummies(df[feature_cols_num + cat_cols], columns=cat_cols)

y = pd.DataFrame(index=df.index)
y['Leakage_96h_log'] = np.log1p(df['Leakage_96h'])
y['Leakage_168h_log'] = np.log1p(df['Leakage_168h'])
y['Resistance_96h_delta'] = df['Resistance_96h'] - df['Resistance_24h']
y['Resistance_168h_delta'] = df['Resistance_168h'] - df['Resistance_24h']
y['Vth_96h_delta'] = df['Vth_96h'] - df['Vth_24h']
y['Vth_168h_delta'] = df['Vth_168h'] - df['Vth_24h']

def reconstruct(raw_preds, idx):
    out = pd.DataFrame(index=idx)
    out['Leakage_96h'] = np.expm1(raw_preds[:, 0])
    out['Leakage_168h'] = np.expm1(raw_preds[:, 1])
    out['Resistance_96h'] = df.loc[idx, 'Resistance_24h'].values + raw_preds[:, 2]
    out['Resistance_168h'] = df.loc[idx, 'Resistance_24h'].values + raw_preds[:, 3]
    out['Vth_96h'] = df.loc[idx, 'Vth_24h'].values + raw_preds[:, 4]
    out['Vth_168h'] = df.loc[idx, 'Vth_24h'].values + raw_preds[:, 5]
    return out[target_cols_raw]

X_train, X_test = X.loc[idx_train], X.loc[idx_test]
y_train, y_test = y.loc[idx_train], y.loc[idx_test]
y_test_actual = df.loc[idx_test, target_cols_raw]
X.shape

## Baseline

In [ ]:
def baseline_predict(idx):
    preds = pd.DataFrame(index=idx)
    for m in METRICS:
        v0 = df.loc[idx, f'{m}_0h']
        v24 = df.loc[idx, f'{m}_24h']
        slope = (v24 - v0) / df.loc[idx, 'elapsed_24h']
        preds[f'{m}_96h'] = v24 + slope * (df.loc[idx, 'elapsed_96h'] - df.loc[idx, 'elapsed_24h'])
        preds[f'{m}_168h'] = v24 + slope * (df.loc[idx, 'elapsed_168h'] - df.loc[idx, 'elapsed_24h'])
    return preds[target_cols_raw]

baseline_preds = baseline_predict(idx_test)

def evaluate(y_true, y_pred, name):
    print(f'--- {name} ---')
    for col in target_cols_raw:
        mae = mean_absolute_error(y_true[col], y_pred[col])
        r2 = r2_score(y_true[col], y_pred[col])
        print(f'{col:16s} MAE={mae:.4f}   R2={r2:.4f}')
    print()

evaluate(y_test_actual, baseline_preds, 'Baseline')

## RF + GB, tuned

same tuning approach as before

In [ ]:
rf_grid = {'n_estimators': [150, 250, 350], 'max_depth': [15, 25, None], 'min_samples_leaf': [1, 2, 4]}
rf_search = RandomizedSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), rf_grid,
                                n_iter=8, cv=2, scoring='neg_mean_absolute_error', random_state=42, n_jobs=-1)
rf_search.fit(X_train, y_train)
rf_tuned = rf_search.best_estimator_
rf_tuned_preds = reconstruct(rf_tuned.predict(X_test), idx_test)
print('best RF params:', rf_search.best_params_)
evaluate(y_test_actual, rf_tuned_preds, 'Random Forest (tuned)')

In [ ]:
gb_grid = {'estimator__n_estimators': [100, 200], 'estimator__learning_rate': [0.05, 0.1], 'estimator__max_depth': [2, 3, 4]}
gb_search = RandomizedSearchCV(MultiOutputRegressor(GradientBoostingRegressor(random_state=42)), gb_grid,
                                n_iter=8, cv=2, scoring='neg_mean_absolute_error', random_state=42, n_jobs=-1)
gb_search.fit(X_train, y_train)
gb_tuned = gb_search.best_estimator_
gb_tuned_preds = reconstruct(gb_tuned.predict(X_test), idx_test)
print('best GB params:', gb_search.best_params_)
evaluate(y_test_actual, gb_tuned_preds, 'Gradient Boosting (tuned)')

## Tried: weighting Fail/Borderline samples higher

Idea was to make the model care more about getting risky parts right, since that matters more
than shaving error off parts that were already obviously fine. Tested a few weight ratios.

Turns out it doesn't help, even at aggressive ratios (1:5:15) - Fail error barely moves and Safe
error actually gets worse. Best guess why: a chunk of the Fail cases are the step-change failures,
and those aren't predictable from 0h/24h no matter how much the model is told to try harder on them.
You can't upweight your way past "the information just isn't in the input". Not using this in the
final model, but leaving the test in since it's a real result worth knowing.

In [ ]:
weight_options = [
    {'Safe': 1, 'Borderline': 2, 'Fail': 3},
    {'Safe': 1, 'Borderline': 5, 'Fail': 15},
]

for weights in weight_options:
    sw = label_train.map(weights).values
    rf_w = RandomForestRegressor(n_estimators=250, max_depth=25, min_samples_leaf=2, random_state=42, n_jobs=-1)
    rf_w.fit(X_train, y_train, sample_weight=sw)
    preds = reconstruct(rf_w.predict(X_test), idx_test)

    check = pd.DataFrame({'label': label_test.values, 'true': y_test_actual['Leakage_168h'].values, 'pred': preds['Leakage_168h'].values})
    check['err'] = (check['true'] - check['pred']).abs()
    print('weights:', weights)
    print(check.groupby('label')['err'].mean())
    print('overall MAE:', round(mean_absolute_error(y_test_actual['Leakage_168h'], preds['Leakage_168h']), 4))
    print()

## Stacked ensemble

Instead of just averaging RF and GB 50/50, train a small Ridge model to learn the best way to
blend them per target. Using out-of-fold predictions on the training set so the meta-model isn't
just cheating by seeing predictions on data it already trained on.

In [ ]:
kf = KFold(n_splits=4, shuffle=True, random_state=42)

rf_base = RandomForestRegressor(n_estimators=250, max_depth=25, min_samples_leaf=2, random_state=42, n_jobs=-1)
gb_base = MultiOutputRegressor(GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.08, random_state=42))

rf_oof = cross_val_predict(rf_base, X_train, y_train, cv=kf, n_jobs=-1)
gb_oof = cross_val_predict(gb_base, X_train, y_train, cv=kf, n_jobs=-1)

meta_X_train = np.hstack([rf_oof, gb_oof])
meta_model = Ridge(alpha=1.0)
meta_model.fit(meta_X_train, y_train)

rf_base.fit(X_train, y_train)
gb_base.fit(X_train, y_train)
meta_X_test = np.hstack([rf_base.predict(X_test), gb_base.predict(X_test)])
stacked_raw = meta_model.predict(meta_X_test)

stacked_preds = reconstruct(stacked_raw, idx_test)
simple_avg_preds = reconstruct((rf_tuned.predict(X_test) + gb_tuned.predict(X_test)) / 2, idx_test)

evaluate(y_test_actual, simple_avg_preds, 'Simple average ensemble')
evaluate(y_test_actual, stacked_preds, 'Stacked ensemble (Ridge)')

Stacking wins on every target, small but consistent (~3-5% lower MAE than the simple average).
Going with this as the final point-estimate model.

## Prediction intervals (quantile regression)

Instead of one number, give a range: "probably between X and Y". Training separate models for the
5th and 95th percentile so we get a 90% interval. This should also naturally widen on parts that
are harder to predict - checking that below.

In [ ]:
gb_lower = MultiOutputRegressor(GradientBoostingRegressor(loss='quantile', alpha=0.05, n_estimators=150, max_depth=3, random_state=42))
gb_upper = MultiOutputRegressor(GradientBoostingRegressor(loss='quantile', alpha=0.95, n_estimators=150, max_depth=3, random_state=42))
gb_lower.fit(X_train, y_train)
gb_upper.fit(X_train, y_train)

lower_preds = reconstruct(gb_lower.predict(X_test), idx_test)
upper_preds = reconstruct(gb_upper.predict(X_test), idx_test)

# coverage check - how often does the true value actually fall inside the interval
for col in target_cols_raw:
    inside = (y_test_actual[col] >= lower_preds[col]) & (y_test_actual[col] <= upper_preds[col])
    print(f'{col:16s} coverage={inside.mean():.1%}  avg width={ (upper_preds[col]-lower_preds[col]).mean():.4f}')

In [ ]:
# does the interval actually widen on the hard-to-predict (step-change-like) rows?
df_test = df.loc[idx_test].copy()
df_test['res_step'] = (df_test['Resistance_168h'] - df_test['Resistance_96h']).abs()
typical_step = df.loc[idx_train, 'Resistance_96h'].sub(df.loc[idx_train, 'Resistance_24h']).abs().median()
suspicious = (df_test['res_step'] > 10 * typical_step).values

width = (upper_preds['Leakage_168h'] - lower_preds['Leakage_168h']).values
print('avg interval width, normal rows:', round(width[~suspicious].mean(), 3))
print('avg interval width, suspicious/step-change rows:', round(width[suspicious].mean(), 3))

Good - the interval is noticeably wider on the rows that look like they had a sudden jump. That's
the model honestly saying "I'm less sure about this one" instead of confidently giving a number
that's probably wrong. This is arguably more useful for the actual use case than a slightly lower
MAE would be - an engineer can see "wide interval = treat this prediction with caution" directly.

## Bonus: other models (XGBoost, LightGBM, neural nets)

Same disclaimer as before - couldn't test these in the environment I built this in
(no internet / no GPU libs installed). Should run on colab, skips cleanly if a library
isn't available. Numbers here are NOT verified by me, check them yourself when you run it.

In [ ]:
# !pip install xgboost lightgbm -q
extra_models = {}

try:
    from xgboost import XGBRegressor
    xgb_model = MultiOutputRegressor(XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.08, random_state=42))
    xgb_model.fit(X_train, y_train)
    extra_models['XGBoost'] = reconstruct(xgb_model.predict(X_test), idx_test)
    evaluate(y_test_actual, extra_models['XGBoost'], 'XGBoost')
except ImportError:
    print('xgboost not installed, skipping')

try:
    from lightgbm import LGBMRegressor
    lgbm_model = MultiOutputRegressor(LGBMRegressor(n_estimators=300, max_depth=6, learning_rate=0.08, random_state=42, verbose=-1))
    lgbm_model.fit(X_train, y_train)
    extra_models['LightGBM'] = reconstruct(lgbm_model.predict(X_test), idx_test)
    evaluate(y_test_actual, extra_models['LightGBM'], 'LightGBM')
except ImportError:
    print('lightgbm not installed, skipping')

### Neural nets

Two versions: a plain dense network (MLP) using the same tabular features as RF/GB, and an LSTM
that takes the 0h/24h readings as a 2-step sequence plus the other features tacked on afterward.

Honestly not expecting the LSTM to do much here - 2 timesteps is really short for a model that's
built to find patterns across long sequences, and this dataset (under 10k rows) is on the small
side for deep learning in general. Including it because it's worth actually checking instead of
just assuming, but the tree models are the ones with something to prove against.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras import Model, Input, Sequential, layers, callbacks

    early_stop = callbacks.EarlyStopping(patience=8, restore_best_weights=True)

    # --- plain dense network on the same features as RF/GB ---
    mlp = Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(32, activation='relu'),
        layers.Dense(6),
    ])
    mlp.compile(optimizer='adam', loss='mae')
    mlp.fit(X_train.values, y_train.values, epochs=100, batch_size=32,
            validation_split=0.15, callbacks=[early_stop], verbose=0)

    mlp_preds = reconstruct(mlp.predict(X_test.values, verbose=0), idx_test)
    extra_models['MLP'] = mlp_preds
    evaluate(y_test_actual, mlp_preds, 'MLP (dense NN)')
    have_tf = True
except ImportError:
    print('tensorflow not installed, skipping neural net models')
    have_tf = False

In [ ]:
if have_tf:
    # sequence input: [Leakage,Resistance,Vth] at 0h then at 24h -> shape (n, 2, 3)
    def build_sequence(idx):
        t0 = df.loc[idx, [f'{m}_0h' for m in METRICS]].values
        t24 = df.loc[idx, [f'{m}_24h' for m in METRICS]].values
        return np.stack([t0, t24], axis=1).astype('float32')

    seq_train = build_sequence(idx_train)
    seq_test = build_sequence(idx_test)

    seq_input = Input(shape=(2, 3), name='sequence')
    lstm_out = layers.LSTM(16)(seq_input)

    static_input = Input(shape=(X_train.shape[1],), name='static')
    combined = layers.concatenate([lstm_out, static_input])
    dense = layers.Dense(32, activation='relu')(combined)
    output = layers.Dense(6)(dense)

    lstm_model = Model(inputs=[seq_input, static_input], outputs=output)
    lstm_model.compile(optimizer='adam', loss='mae')
    lstm_model.fit([seq_train, X_train.values], y_train.values, epochs=100, batch_size=32,
                   validation_split=0.15, callbacks=[early_stop], verbose=0)

    lstm_preds = reconstruct(lstm_model.predict([seq_test, X_test.values], verbose=0), idx_test)
    extra_models['LSTM'] = lstm_preds
    evaluate(y_test_actual, lstm_preds, 'LSTM')

## Full comparison

In [ ]:
all_preds = {
    'Baseline': baseline_preds,
    'RF (tuned)': rf_tuned_preds,
    'GB (tuned)': gb_tuned_preds,
    'Simple avg ensemble': simple_avg_preds,
    'Stacked ensemble': stacked_preds,
}
all_preds.update(extra_models)

results = []
for name, preds in all_preds.items():
    for col in target_cols_raw:
        mae = mean_absolute_error(y_test_actual[col], preds[col])
        results.append({'model': name, 'target': col, 'MAE': mae})

comparison = pd.DataFrame(results).pivot(index='target', columns='model', values='MAE')
comparison = comparison[list(all_preds.keys())]
comparison

In [ ]:
comparison.plot(kind='bar', figsize=(14,5))
plt.ylabel('MAE (lower = better)')
plt.title('All models compared')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()

**So far (verified on my end): Stacked ensemble is the best point-estimate model**, winning or
tying on every target. If XGBoost/LightGBM/the neural nets beat it when you run this - that's a
legit result too, just wasn't something I could confirm myself given the environment I built this in.

## Which features matter (from RF)

In [ ]:
importances = pd.Series(rf_tuned.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10)

In [ ]:
importances.head(10).plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Top 10 features - Random Forest')
plt.tight_layout()
plt.show()

## Error by label

In [ ]:
check = pd.DataFrame({
    'Label': label_test.values,
    'true': y_test_actual['Leakage_168h'].values,
    'pred': stacked_preds['Leakage_168h'].values,
})
check['abs_error'] = (check['true'] - check['pred']).abs()
check.groupby('Label')['abs_error'].mean().sort_values(ascending=False)

## Save everything

Saving the stacking pieces (RF, GB, Ridge meta-model), the quantile models for intervals, and the
imputer. Run new data through the imputer first before predicting.

In [ ]:
import joblib
joblib.dump({
    'rf': rf_base,
    'gb': gb_base,
    'meta_model': meta_model,
    'quantile_lower': gb_lower,
    'quantile_upper': gb_upper,
    'imputer': imputer,
    'feature_cols_raw': feature_cols_raw,
    'feature_columns': list(X.columns),
}, 'drift_prediction_models.joblib')
print('saved')

## Using this with a real ISRO dataset

Should mostly just work if the real CSV uses the same column names. If not:

1. Remap columns to match the CONFIG cell (Leakage_0h etc, Timestamp_0h/24h/96h/168h, Proxy_Stress,
   Stress_Level, Part_Type, Label)
2. If real checkpoints aren't 0h/24h/96h/168h, update CHECKPOINTS/FEATURE_CHECKPOINTS/TARGET_CHECKPOINTS
   at the top - everything downstream reads from those
3. Re-run top to bottom, including the tuning cells (params found on synthetic data may not transfer)

Expect real Fail rates to be much lower than the ~15-17% used here for the demo - real qualification
data is usually far more imbalanced. If Fail examples are rare, treat this notebook's accuracy numbers
as optimistic and lean more on the Dynamic Anomaly side of the pipeline for those rare cases, since
it doesn't need labeled failures to work in the first place.